# 7단계: 날씨 direct feature 비교 단계

## 이 단계의 핵심 변화
날씨를 직접 feature로 넣는 여러 방식을 비교한 단계입니다.

## 왜 점수가 좋아졌나
외부 데이터가 도움이 될 가능성은 보였지만 direct feature 방식은 public에서 불안정했습니다.

## 안내
이 파일은 다운로드 오류를 피하기 위해 새 이름으로 다시 만든 버전입니다.
코드 셀 안에는 기존 주석이 남아 있거나, 원본 설명 흐름을 유지합니다.


# 현재 최고 구조 + 날씨 변수(기온/강수) 비교 v2

목표:
- 현재 최고 구조인 **참여율 기반 + 중식/석식 피처 분리 + weak correction(0.35/0.35/clip 0.04)** 유지
- 정리된 날씨 파일(`weather_no_snowfall.csv`)을 사용해
  **기온/강수 변수만 가볍게 추가**해서 validation 비교

비교 버전:
- `baseline_best`
- `add_temp_basic`
- `add_rain_basic`
- `add_temp_rain_basic`

사용 날씨 파일 우선순위:
1. `C:\Users\yuzhd\github\team\날씨\weather.csv`
2. `/mnt/data/weather_no_snowfall.csv`
3. 기타 후보 경로

In [ ]:
# 필요시 한 번만 설치
# !pip install xgboost scikit-learn

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# 1. 파일 경로 자동 탐색
train_candidates = [
    r"C:\Users\yuzhd\github\team\data\made\train_median.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/made/train_median.csv",
    r"/mnt/data/train_median.csv",
    r"./train_median.csv",
]

weather_candidates = [
    r"C:\Users\yuzhd\github\team\날씨\weather.csv",
    r"C:\Users\yuzhd\github\team\날씨\weather_no_snowfall.csv",
    r"/mnt/c/Users/yuzhd/github/team/날씨/weather.csv",
    r"/mnt/c/Users/yuzhd/github/team/날씨/weather_no_snowfall.csv",
    r"/mnt/data/weather_no_snowfall.csv",
    r"/mnt/data/weather.csv",
    r"./weather_no_snowfall.csv",
    r"./weather.csv",
]

def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

train_path = find_existing_path(train_candidates)
weather_path = find_existing_path(weather_candidates)

print("train path:", train_path)
print("weather path:", weather_path)

if train_path is None:
    raise FileNotFoundError("train_median.csv 파일을 찾지 못했습니다.")
if weather_path is None:
    raise FileNotFoundError("weather_no_snowfall.csv 또는 weather.csv 파일을 찾지 못했습니다.")

In [ ]:
# 2. 데이터 로드
df = pd.read_csv(train_path, encoding="utf-8-sig")
weather = pd.read_csv(weather_path, encoding="utf-8-sig")

df["일자"] = pd.to_datetime(df["일자"])

print("[train]")
display(df.head())

print("[weather]")
display(weather.head())

print("[weather columns]")
print(list(weather.columns))

In [ ]:
# 3. 날씨 최소 전처리
# - 날짜 통일
# - 강수량 결측 0 처리
# - 기온 보간
# - 적설 컬럼은 이미 제거되었거나 무시

def find_col(cols, candidates):
    for cand in candidates:
        for c in cols:
            if cand.lower() == c.lower():
                return c
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

date_col = find_col(weather.columns, ["일시", "date", "날짜"])
temp_col = find_col(weather.columns, ["기온", "평균기온", "temperature", "avg_temp"])
rain_col = find_col(weather.columns, ["강수량", "rainfall", "precipitation", "rain"])

if date_col is None:
    raise ValueError("날씨 파일에서 날짜 컬럼을 찾지 못했습니다.")
if temp_col is None:
    raise ValueError("날씨 파일에서 기온 컬럼을 찾지 못했습니다.")
if rain_col is None:
    raise ValueError("날씨 파일에서 강수 컬럼을 찾지 못했습니다.")

weather = weather[[date_col, temp_col, rain_col]].copy()
weather = weather.rename(columns={date_col: "일자", temp_col: "기온", rain_col: "강수량"})
weather["일자"] = pd.to_datetime(weather["일자"])
weather["기온"] = pd.to_numeric(weather["기온"], errors="coerce")
weather["강수량"] = pd.to_numeric(weather["강수량"], errors="coerce")

weather["강수량"] = weather["강수량"].fillna(0)
weather["기온"] = weather["기온"].interpolate().bfill().ffill()

weather["is_rain"] = (weather["강수량"] > 0).astype(int)
weather["temp_sq"] = weather["기온"] ** 2
weather["is_hot"] = (weather["기온"] >= 28).astype(int)
weather["is_cold"] = (weather["기온"] <= 5).astype(int)

print("[정리된 weather]")
display(weather.head())
print(weather.isna().sum())

In [ ]:
# 4. merge
df = df.merge(weather, on="일자", how="left")

for c in ["기온", "강수량", "is_rain", "temp_sq", "is_hot", "is_cold"]:
    if c in df.columns:
        if c in ["is_rain", "is_hot", "is_cold"]:
            df[c] = df[c].fillna(0)
        else:
            df[c] = df[c].interpolate().bfill().ffill()

display(df[["일자", "기온", "강수량", "is_rain", "temp_sq", "is_hot", "is_cold"]].head())

In [ ]:
# 5. 기본 피처 생성
def normalize_menu_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\n", " ").replace("\r", " ").replace("/", " ").replace("&", " ").replace("+", " ")
    text = text.replace("(", " ").replace(")", " ")
    text = re.sub(r"[^가-힣A-Za-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def add_features(df):
    df = df.copy()

    df["월"] = df["일자"].dt.month
    df["일"] = df["일자"].dt.day
    df["요일"] = df["일자"].dt.weekday

    df["식사가능자수"] = (
        df["본사정원수"]
        - df["본사휴가자수"]
        - df["본사출장자수"]
        - df["현본사소속재택근무자수"]
    ).clip(lower=1)

    month_end = df["일자"] + pd.offsets.MonthEnd(0)
    df["days_to_month_end"] = (month_end - df["일자"]).dt.days
    df["is_month_end_part"] = (df["days_to_month_end"] <= 5).astype(int)
    df["is_wed"] = (df["요일"] == 2).astype(int)
    df["is_fri"] = (df["요일"] == 4).astype(int)

    df["has_overtime"] = (df["본사시간외근무명령서승인건수"] > 0).astype(int)
    df["log_overtime"] = np.log1p(df["본사시간외근무명령서승인건수"])

    return df

df = add_features(df)
df["중식참여율"] = (df["중식계"] / df["식사가능자수"]).clip(lower=0, upper=1.5)
df["석식참여율"] = (df["석식계"] / df["식사가능자수"]).clip(lower=0, upper=1.5)

In [ ]:
# 6. 현재 최고 기본 피처
base_lunch = [
    "월","일","요일","식사가능자수",
    "본사출장자수","본사시간외근무명령서승인건수",
    "is_fri","days_to_month_end","is_month_end_part",
]

base_dinner = [
    "월","일","요일","식사가능자수",
    "본사출장자수","본사시간외근무명령서승인건수",
    "is_wed","has_overtime","log_overtime",
]

In [ ]:
# 7. weak correction용 메뉴 플래그
lunch_keyword_map = {
    "chicken":"치킨",
    "donkatsu":"돈까스",
    "jeyuk":"제육",
    "curry":"카레",
    "fried_rice":"볶음밥",
    "bibimbap":"비빔밥",
    "jjajang":"짜장",
    "tangsuyuk":"탕수육",
}

def lunch_keyword_flags(text):
    text = normalize_menu_text(text)
    return {f"kw_{k}": int(v in text) for k, v in lunch_keyword_map.items()}

def dinner_style_flags(text):
    text = normalize_menu_text(text)
    flags = {
        "style_korean":0,
        "style_chinese":0,
        "style_western":0,
        "style_japanese":0,
        "style_snack":0,
        "style_fusion":0,
    }
    chinese_kw = ["짜장","짬뽕","탕수","마파","깐풍","유산슬"]
    western_kw = ["돈까스","스테이크","파스타","스프","햄버거","피자","리조또"]
    japanese_kw = ["초밥","우동","가라아게","돈부리","소바"]
    snack_kw = ["떡볶이","순대","튀김","라볶이","김밥","분식","어묵"]
    fusion_kw = ["브리또","타코","샐러드","또띠아","퓨전"]

    if any(k in text for k in chinese_kw): flags["style_chinese"] = 1
    if any(k in text for k in western_kw): flags["style_western"] = 1
    if any(k in text for k in japanese_kw): flags["style_japanese"] = 1
    if any(k in text for k in snack_kw): flags["style_snack"] = 1
    if any(k in text for k in fusion_kw): flags["style_fusion"] = 1
    if sum(flags.values()) == 0: flags["style_korean"] = 1
    return flags

lunch_menu = pd.DataFrame([lunch_keyword_flags(x) for x in df["중식메뉴"].fillna("")])
dinner_menu = pd.DataFrame([dinner_style_flags(x) for x in df["석식메뉴"].fillna("")])

In [ ]:
# 8. 버전 정의
weather_temp_basic = ["기온", "temp_sq", "is_hot", "is_cold"]
weather_rain_basic = ["강수량", "is_rain"]

versions = {
    "baseline_best": {
        "lunch": base_lunch,
        "dinner": base_dinner,
    },
    "add_temp_basic": {
        "lunch": base_lunch + weather_temp_basic,
        "dinner": base_dinner + weather_temp_basic,
    },
    "add_rain_basic": {
        "lunch": base_lunch + weather_rain_basic,
        "dinner": base_dinner + weather_rain_basic,
    },
    "add_temp_rain_basic": {
        "lunch": base_lunch + weather_temp_basic + weather_rain_basic,
        "dinner": base_dinner + weather_temp_basic + weather_rain_basic,
    },
}
versions

In [ ]:
# 9. validation split
split_date = pd.Timestamp("2020-11-01")
train_idx = df["일자"] < split_date
valid_idx = df["일자"] >= split_date

train_df = df.loc[train_idx].copy().reset_index(drop=True)
valid_df = df.loc[valid_idx].copy().reset_index(drop=True)

train_lunch_menu = lunch_menu.loc[train_idx].reset_index(drop=True)
valid_lunch_menu = lunch_menu.loc[valid_idx].reset_index(drop=True)

train_dinner_menu = dinner_menu.loc[train_idx].reset_index(drop=True)
valid_dinner_menu = dinner_menu.loc[valid_idx].reset_index(drop=True)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def nmae_fixed(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred) / (np.mean(y_true) + 1e-8) * 100

def evaluate_regression(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "NMAE_fixed(%)": nmae_fixed(y_true, y_pred),
    }

In [ ]:
# 10. weak correction 고정값
params = {
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 5,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 1,
    "gamma": 0.0,
}
LUNCH_SHRINK = 0.35
DINNER_SHRINK = 0.35
RAW_CLIP = 0.03
FINAL_CLIP = 0.04

In [ ]:
# 11. 버전별 비교
rows = []

for vname, feat in versions.items():
    lunch_features = feat["lunch"]
    dinner_features = feat["dinner"]

    X_train_lunch = train_df[lunch_features]
    X_valid_lunch = valid_df[lunch_features]
    y_train_lunch = train_df["중식참여율"]
    y_valid_lunch_count = valid_df["중식계"].values

    X_train_dinner = train_df[dinner_features]
    X_valid_dinner = valid_df[dinner_features]
    y_train_dinner = train_df["석식참여율"]
    y_valid_dinner_count = valid_df["석식계"].values

    lunch_model = XGBRegressor(objective="reg:squarederror", random_state=42, **params)
    dinner_model = XGBRegressor(objective="reg:squarederror", random_state=42, **params)

    lunch_model.fit(X_train_lunch, y_train_lunch)
    dinner_model.fit(X_train_dinner, y_train_dinner)

    train_pred_lunch_ratio = np.clip(lunch_model.predict(X_train_lunch), 0, 1.5)
    valid_pred_lunch_ratio = np.clip(lunch_model.predict(X_valid_lunch), 0, 1.5)

    train_pred_dinner_ratio = np.clip(dinner_model.predict(X_train_dinner), 0, 1.5)
    valid_pred_dinner_ratio = np.clip(dinner_model.predict(X_valid_dinner), 0, 1.5)

    tmp_train = train_df.copy()
    tmp_train["lunch_residual"] = tmp_train["중식참여율"] - train_pred_lunch_ratio
    tmp_train["dinner_residual"] = tmp_train["석식참여율"] - train_pred_dinner_ratio

    lunch_corr = {}
    for col in train_lunch_menu.columns:
        mask = train_lunch_menu[col] == 1
        if mask.sum() >= 5:
            lunch_corr[col] = tmp_train.loc[mask, "lunch_residual"].mean()
        else:
            lunch_corr[col] = 0.0
    for k in lunch_corr:
        lunch_corr[k] = float(np.clip(lunch_corr[k] * LUNCH_SHRINK, -RAW_CLIP, RAW_CLIP))

    dinner_corr = {}
    for col in train_dinner_menu.columns:
        mask = train_dinner_menu[col] == 1
        if mask.sum() >= 5:
            dinner_corr[col] = tmp_train.loc[mask, "dinner_residual"].mean()
        else:
            dinner_corr[col] = 0.0
    for k in dinner_corr:
        dinner_corr[k] = float(np.clip(dinner_corr[k] * DINNER_SHRINK, -RAW_CLIP, RAW_CLIP))

    lunch_adj = np.zeros(len(valid_df))
    for col in valid_lunch_menu.columns:
        lunch_adj += valid_lunch_menu[col].values * lunch_corr.get(col, 0.0)

    dinner_adj = np.zeros(len(valid_df))
    for col in valid_dinner_menu.columns:
        dinner_adj += valid_dinner_menu[col].values * dinner_corr.get(col, 0.0)

    lunch_adj = np.clip(lunch_adj, -FINAL_CLIP, FINAL_CLIP)
    dinner_adj = np.clip(dinner_adj, -FINAL_CLIP, FINAL_CLIP)

    final_lunch_rate = np.clip(valid_pred_lunch_ratio + lunch_adj, 0, 1.5)
    final_dinner_rate = np.clip(valid_pred_dinner_ratio + dinner_adj, 0, 1.5)

    pred_lunch_count = final_lunch_rate * valid_df["식사가능자수"].values
    pred_dinner_count = final_dinner_rate * valid_df["식사가능자수"].values

    rows.append({
        "target": "중식계",
        "version": vname,
        **evaluate_regression(y_valid_lunch_count, pred_lunch_count)
    })
    rows.append({
        "target": "석식계",
        "version": vname,
        **evaluate_regression(y_valid_dinner_count, pred_dinner_count)
    })

result_df = pd.DataFrame(rows).sort_values(["target", "NMAE_fixed(%)", "RMSE"]).reset_index(drop=True)
display(result_df)

In [ ]:
# 12. 결과 저장
result_df.to_csv("weather_feature_compare_result.csv", index=False, encoding="utf-8-sig")
print("저장 완료: weather_feature_compare_result.csv")